In [56]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import joblib

In [57]:
n = 10000

In [58]:
np.random.seed(1)

In [59]:
genomic = pd.DataFrame({
    'Mutation_TP53': np.random.choice([0, 1], n, p = [0.6, 0.4]),
    'Mutation_IDH1': np.random.choice([0, 1], n, p = [0.65, 0.35]),
    'Mutation_EGFR': np.random.choice([0, 1], n, p = [0.7, 0.3])
})

for i in range(1, 6):
    genomic[f'Gene_{i}'] = np.random.randint(0, 101, n)

In [60]:
for i in range(1, 6):
    genomic[f'Gene_{i}'] = np.log2(genomic[f'Gene_{i}'] + 1)

In [61]:
clinical = pd.DataFrame({
    'Age': np.random.randint(18, 80, n),
    'Gender': np.random.choice(['Male', 'Female'], n),
    'KPS': np.random.choice([40, 50, 60, 70, 80, 90, 100], n),
    'Comorbidities': np.random.choice(['None', 'Diabetes', 'Hypertension', 'Heart Disease', 'Multiple'], n),
})

In [62]:
laboratory = pd.DataFrame({
    "WBC": np.random.randint(3500, 15001, n),
    "CRP": np.round(np.random.uniform(0, 30, n), 1),
    "Albumin": np.round(np.random.uniform(2.5, 5.0, n), 1)
})

In [63]:
treatment = pd.DataFrame({
    'Drug': np.random.choice(['Temozolomide', 'Lomustine', 'Bevacizumab', 'Carmustine', 'PCV'],n),
    'Dose': np.random.uniform(50, 301, n),
    'Route': np.random.choice(['Oral', 'IV', 'Intrathecal'], n)
})

In [64]:
risk_score = (
      1.3 * genomic['Mutation_TP53']
    - 1.5 * genomic['Mutation_IDH1']
    + 0.9 * genomic['Mutation_EGFR']
    + 0.12 * genomic['Gene_1']
    - 0.08 * genomic['Gene_2']
    + 0.05 * genomic['Gene_3']
    + 0.10 * genomic['Gene_4']
    - 0.06 * genomic['Gene_5']
    
    + 0.05 * clinical['Age']
    - 0.06 * clinical['KPS']
    + 0.7 * clinical['Gender'].map({'Male': 1,'Female': 0})
    + 0.6 * clinical['Comorbidities'].map({'None': 0,'Diabetes': 1,'Hypertension': 2,
                                           'Heart Disease': 3,'Multiple': 4})

    + 0.0003 * laboratory['WBC']
    + 0.08 * laboratory['CRP']
    - 0.8 * laboratory['Albumin']

    
    + 0.05 * treatment['Route'].map({'Oral': 1, 'IV': 2, 'Intrathecal': 2})
)

risk_score = ((risk_score - risk_score.mean()) / risk_score.std())

In [65]:
toxicity_score = 1.2 * risk_score + np.random.normal(0, 0.4, n)

severe_toxicity = (toxicity_score > 0).astype(int)

In [66]:
response_score = (-2.2 * risk_score
                  + (0.003 * treatment['Dose'] *
                     treatment['Drug'].map({'Temozolomide': 0.8,'Lomustine': 1.0,
                                            'Bevacizumab': 0.6,'Carmustine': 1.2,'PCV': 1.4}))
                 - np.random.normal(0, 0.12, n))

treatment_response = pd.cut(
    response_score,
    bins=[-np.inf, -1, 0, 1, np.inf],
    labels=['PD', 'SD', 'PR', 'CR']
)

In [67]:
true_survival_time = (
    24
    - 6 * risk_score
    - 3 * severe_toxicity
    + np.random.normal(0, 2.8, n)
)

true_survival_time = np.clip(true_survival_time, 1, 36)

event_occured = (true_survival_time <= 24).astype(int)

In [68]:
for col in ['Gene_1', 'Gene_3', 'Gene_4']:
    genomic.loc[genomic.sample(frac = 0.02, random_state = 1).index, col] = round(genomic[col].mean() * 10)

In [69]:
inconsistents = clinical.sample(frac = 0.05, random_state = 2).index
clinical.loc[inconsistents, 'Gender'] = np.random.choice([
    'male', 'MALE', 'M', 'female', 'FEMALE', 'F'], len(inconsistents))

In [70]:
for col in ['WBC', 'CRP']:
    laboratory.loc[laboratory.sample(frac = 0.05, random_state = 3).index, col] = np.nan

In [71]:
dups = treatment.sample(frac = 0.05, random_state = 4)
treatment = pd.concat([treatment, dups], axis = 0).reset_index(drop = True)

In [72]:
gender_map = {
    'Male': 'Male', 'male': 'Male', 'MALE': 'Male', 'M': 'Male',
    'Female': 'Female', 'female': 'Female', 'FEMALE': 'Female', 'F': 'Female'
}

clinical['Gender'] = clinical['Gender'].map(gender_map)

In [73]:
treatment = treatment.drop_duplicates()

In [74]:
clinical_noms = ['Gender']
clinical_ords = ['Comorbidities']

In [75]:
encoded = [['None', 'Diabetes', 'Hypertension', 'Heart Disease', 'Multiple']]

In [76]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])

In [77]:
nom_pipe = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

In [78]:
ord_pipe = Pipeline([
    ('encoder', OrdinalEncoder(categories = encoded))
])

In [79]:
genomic_pp = ColumnTransformer([
    ('num', num_pipe, genomic.columns.tolist())
])

In [80]:
clinical_pp = ColumnTransformer([
    ('num', num_pipe, clinical.select_dtypes(include = 'number').columns.tolist()),
    ('nom', nom_pipe, clinical_noms),
    ('ord', ord_pipe, clinical_ords)
])

In [81]:
laboratory_pp = ColumnTransformer([
    ('num', num_pipe, laboratory.columns.tolist())
])

In [82]:
treatment_pp = ColumnTransformer([
    ('num', num_pipe, treatment.select_dtypes(include = 'number').columns.tolist()),
    ('nom', nom_pipe, treatment.select_dtypes(exclude = 'number').columns.tolist())
])

In [83]:
(x_gen_train, x_gen_test,
 x_clinic_train, x_clinic_test,
 x_lab_train, x_lab_test,
 x_treat_train, x_treat_test,
 y_survive_train, y_survive_test,
 y_response_train, y_response_test,
 y_toxic_train, y_toxic_test) = train_test_split(
     genomic, clinical, laboratory, treatment,
     event_occured, treatment_response, severe_toxicity,
     test_size = 0.2, random_state = 5)

In [84]:
for col in genomic.columns.tolist():
    Q1 = x_gen_train[col].quantile(0.25)
    Q3 = x_gen_train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    x_gen_train[col] = x_gen_train[col].clip(lower, upper)
    x_gen_test[col] = x_gen_test[col].clip(lower, upper)

In [85]:
le = LabelEncoder()
y_response_train = le.fit_transform(y_response_train)
y_response_test = le.transform(y_response_test)

In [86]:
x_gen_train = genomic_pp.fit_transform(x_gen_train)
x_gen_test = genomic_pp.transform(x_gen_test)

In [87]:
x_clinic_train = clinical_pp.fit_transform(x_clinic_train)
x_clinic_test = clinical_pp.transform(x_clinic_test)

In [88]:
x_lab_train = laboratory_pp.fit_transform(x_lab_train)
x_lab_test = laboratory_pp.transform(x_lab_test)

In [89]:
x_treat_train = treatment_pp.fit_transform(x_treat_train)
x_treat_test = treatment_pp.transform(x_treat_test)

In [90]:
input1 = Input(shape = (x_gen_train.shape[1], ), name = 'genomic')
input2 = Input(shape = (x_clinic_train.shape[1], ), name = 'clinical')
input3 = Input(shape = (x_lab_train.shape[1], ), name = 'laboratory')
input4 = Input(shape = (x_treat_train.shape[1], ), name = 'treatment')

a = Dense(64, activation = 'relu')(input1)
ab = BatchNormalization()(a)
ad = Dropout(0.2)(ab)

b = Dense(64, activation = 'relu')(input2)
bb = BatchNormalization()(b)
bd = Dropout(0.2)(bb)

c = Dense(64, activation = 'relu')(input3)
cb = BatchNormalization()(c)
cd = Dropout(0.2)(cb)

d = Dense(64, activation = 'relu')(input4)
db = BatchNormalization()(d)
dd = Dropout(0.2)(db)

merged = Concatenate()([ad, bd, cd, dd])

final = Dense(128, activation = 'relu')(merged)
final_d = Dropout(0.4)(final)

survival_out = Dense(1, activation = 'sigmoid', name = 'survival')(final_d)

response_out = Dense(4, activation = 'softmax', name = 'response')(final_d)

toxicity_out = Dense(1, activation = 'sigmoid', name = 'toxicity')(final_d)

In [91]:
model = Model(
    inputs = [input1, input2, input3, input4],
    outputs = [survival_out, response_out, toxicity_out]
)

In [92]:
model.compile(
    optimizer = 'adam',
    loss = {'survival': 'binary_crossentropy',
            'response': 'sparse_categorical_crossentropy',
            'toxicity': 'binary_crossentropy'},
    
    loss_weights = {'survival': 1,
                    'response': 2,
                    'toxicity': 1},
    
    metrics = {'survival': ['accuracy'],
              'response': ['accuracy'],
              'toxicity': ['accuracy']}
)

In [93]:
early_stop = EarlyStopping(
    monitor = 'val_loss',
    patience = 15,
    restore_best_weights = True
)

In [94]:
model.fit(
    [x_gen_train, x_clinic_train, x_lab_train, x_treat_train],
    {'survival': y_survive_train, 'response': y_response_train, 'toxicity': y_toxic_train},
    callbacks = [early_stop],
    epochs = 50,
    batch_size = 64,
    validation_split = 0.1,
    verbose = 1
)

Epoch 1/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 2.9150 - response_accuracy: 0.6099 - response_loss: 0.9896 - survival_accuracy: 0.7660 - survival_loss: 0.4642 - toxicity_accuracy: 0.7744 - toxicity_loss: 0.4679 - val_loss: 2.2771 - val_response_accuracy: 0.7487 - val_response_loss: 0.7532 - val_survival_accuracy: 0.8725 - val_survival_loss: 0.3860 - val_toxicity_accuracy: 0.8612 - val_toxicity_loss: 0.3716
Epoch 2/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.7964 - response_accuracy: 0.7376 - response_loss: 0.5919 - survival_accuracy: 0.8665 - survival_loss: 0.3056 - toxicity_accuracy: 0.8690 - toxicity_loss: 0.3046 - val_loss: 1.6465 - val_response_accuracy: 0.8025 - val_response_loss: 0.5268 - val_survival_accuracy: 0.8875 - val_survival_loss: 0.2925 - val_toxicity_accuracy: 0.8763 - val_toxicity_loss: 0.2845
Epoch 3/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.6454 - response_accuracy: 0.7747 - response_loss: 0.5264 - survival_accuracy: 0.8726 - surviva

In [95]:
results = model.evaluate(
    [x_gen_test, x_clinic_test, x_lab_test, x_treat_test],
    {'survival': y_survive_test, 'response': y_response_test, 'toxicity': y_toxic_test},
    verbose = 0,
    return_dict = True
)

In [96]:
print(results)

{'loss': 1.1337521076202393, 'response_accuracy': 0.8815000057220459, 'response_loss': 0.3210712671279907, 'survival_accuracy': 0.8855000138282776, 'survival_loss': 0.25205332040786743, 'toxicity_accuracy': 0.887499988079071, 'toxicity_loss': 0.2407568246126175}


In [97]:
joblib.dump(model, 'model.joblib')
joblib.dump(genomic_pp, 'genomic_pp.joblib')
joblib.dump(clinical_pp, 'clinical_pp.joblib')
joblib.dump(laboratory_pp, 'laboratory_pp.joblib')
joblib.dump(treatment_pp, 'treatment_pp.joblib')

['treatment_pp.joblib']